# 1. Loading the Processed Dataset

In [4]:
import pandas as pd

df = pd.read_csv('processed_job_data.csv')
df.head()

,standardized_job_title,clean_job_title,company_name,job_country,employment_type,is_remote,job_description,job_source,posted_at_utc,normalized_job_description,extracted_skills,extracted_tools,extracted_technologies
0,data scientist,data scientist - open to all nationalities,Ajman University,AE,full-time,False,SUMMARY OF FUNCTIONS:\n\nThe Data Scientist su...,indeed,2026-04-23 00:00:00+00:00,summary of functions:\n\nthe data scientist su...,"['python', 'r']",['tableau'],['machine learning']
1,data scientist,"senior data scientist united arab emirates, dubai",Dataiku,AE,internship,False,"OverviewDataiku is The Universal AI Platform, ...",bebee,2026-04-25 00:00:00+00:00,"overviewdataiku is the universal ai platform, ...","['python', 'r']",[],['machine learning']
2,data scientist,data scientist-1205,Atribs Metscon,AE,full-time,False,JOB PURPOSE:\n\nLooking for Data & Artificial ...,bayt.com,2026-04-21 00:00:00+00:00,job purpose:\n\nlooking for data & artificial ...,['python'],[],['machine learning']
3,data scientist,junior genomics data scientist (gwas & qc),"United Arab Emirates University, Department of...",AE,full-time,False,A leading academic institution in Abu Dhabi se...,bebee,2026-04-23 00:00:00+00:00,a leading academic institution in abu dhabi se...,"['python', 'r']",[],[]
4,data scientist,data scientist-1205,Atribs Metscon Group,AE,full-time,False,"FAB AL Jaddaf, United Arab Emirates Salary: 23...",bebee,2026-04-25 00:00:00+00:00,"fab al jaddaf, united arab emirates salary: 23...",['python'],[],['machine learning']


## 1.1 Restoring Extracted Columns as Python Lists

In [5]:
df.dtypes

standardized_job_title         str
clean_job_title                str
company_name                   str
job_country                    str
employment_type                str
is_remote                     bool
job_description                str
job_source                     str
posted_at_utc                  str
normalized_job_description     str
extracted_skills               str
extracted_tools                str
extracted_technologies         str
dtype: object

In [7]:
import ast 

df['extracted_skills'] = df['extracted_skills'].apply(ast.literal_eval)
df['extracted_tools'] = df['extracted_tools'].apply(ast.literal_eval)
df['extracted_technologies'] = df['extracted_technologies'].apply(ast.literal_eval)

## 1.2 Verifying Restored Python Object Types

In [13]:
type(df['extracted_skills'][0])

list

## 1.3 Preparing Nested NLP Data for Analysis

### Observation

The extracted NLP columns contained nested list structures where multiple skills/tools/technologies were stored inside a single row.

Using `explode()` transformed each list element into an individual row, converting the data from a nested structure into a tabular format suitable for frequency analysis.

This transformation is necessary because functions like `value_counts()` operate correctly on atomic observations rather than entire Python lists.

In [24]:
df.explode('extracted_skills').head(5)


,standardized_job_title,clean_job_title,company_name,job_country,employment_type,is_remote,job_description,job_source,posted_at_utc,normalized_job_description,extracted_skills,extracted_tools,extracted_technologies
0,data scientist,data scientist - open to all nationalities,Ajman University,AE,full-time,False,SUMMARY OF FUNCTIONS:\n\nThe Data Scientist su...,indeed,2026-04-23 00:00:00+00:00,summary of functions:\n\nthe data scientist su...,python,[tableau],[machine learning]
0,data scientist,data scientist - open to all nationalities,Ajman University,AE,full-time,False,SUMMARY OF FUNCTIONS:\n\nThe Data Scientist su...,indeed,2026-04-23 00:00:00+00:00,summary of functions:\n\nthe data scientist su...,r,[tableau],[machine learning]
1,data scientist,"senior data scientist united arab emirates, dubai",Dataiku,AE,internship,False,"OverviewDataiku is The Universal AI Platform, ...",bebee,2026-04-25 00:00:00+00:00,"overviewdataiku is the universal ai platform, ...",python,[],[machine learning]
1,data scientist,"senior data scientist united arab emirates, dubai",Dataiku,AE,internship,False,"OverviewDataiku is The Universal AI Platform, ...",bebee,2026-04-25 00:00:00+00:00,"overviewdataiku is the universal ai platform, ...",r,[],[machine learning]
2,data scientist,data scientist-1205,Atribs Metscon,AE,full-time,False,JOB PURPOSE:\n\nLooking for Data & Artificial ...,bayt.com,2026-04-21 00:00:00+00:00,job purpose:\n\nlooking for data & artificial ...,python,[],[machine learning]


## 1.4 Frequency Analysis on Exploded NLP Columns

### Observation

- Python is the most frequently requested skill across the analyzed job postings, followed by SQL and R.
- Tableau is the dominant visualization/reporting tool, while Power BI appears less frequently.
- Machine Learning is the most demanded technology area, followed by Deep Learning and NLP.
- These frequency distributions help identify the most in-demand technical competencies in the current job market dataset.
- Frequency analysis provides a foundation for deeper role-based and market trend analysis in later stages of the project.

In [ ]:
skills_count = df.explode('extracted_skills')['extracted_skills'].value_counts()
tool_count = df.explode('extracted_tools')['extracted_tools'].value_counts()
technologies_count = df.explode('extracted_technologies')['extracted_technologies'].value_counts()

In [ ]:
print(skills_count)
print(tool_count)
print(technologies_count)   

extracted_skills
python    36
sql       28
r         12
Name: count, dtype: int64
extracted_tools
tableau    15
powerbi     2
Name: count, dtype: int64
extracted_technologies
machine learning    24
deep learning       10
nlp                  6
Name: count, dtype: int64


## 1.5 Understanding Frequency Distribution Limitations

### Analytical Note

- The extraction pipeline uses `re.search()`, which checks whether a term appears anywhere in the job description.
- Because of this, each skill/tool/technology is extracted only once per job posting, even if it appears multiple times in the same text.
- Therefore, the frequency counts represent job-level demand occurrence, not raw mention frequency.
- This makes the analysis more suitable for job market intelligence because each job posting contributes only one count per extracted term.

## 1.6 Comparing Skill Demand Across Job Roles

### Observation

- Role-based frequency analysis reveals clear specialization patterns between job categories.
- Data Scientist roles show stronger demand for Python, Machine Learning, Deep Learning, and NLP-related technologies.
- Data Analyst roles are more strongly associated with SQL, Tableau, and reporting-oriented skills.
- Segmenting skill demand by job role provides more accurate market intelligence than global frequency analysis alone.
- This analysis helps identify the technical competency profile associated with each role category.

In [41]:
skills_per_job = df.explode('extracted_skills').groupby('standardized_job_title')['extracted_skills'].value_counts()
tools_per_job = df.explode('extracted_tools').groupby('standardized_job_title')['extracted_tools'].value_counts()
technologies_per_job = df.explode('extracted_technologies').groupby('standardized_job_title')['extracted_technologies'].value_counts()

In [45]:
print(skills_per_job)
print(tools_per_job)
print(technologies_per_job)

standardized_job_title  extracted_skills
data analyst            sql                 16
                        python              12
                        r                    5
data scientist          python              24
                        sql                 12
                        r                    7
Name: count, dtype: int64
standardized_job_title  extracted_tools
data analyst            tableau            12
                        powerbi             2
data scientist          tableau             3
Name: count, dtype: int64
standardized_job_title  extracted_technologies
data analyst            machine learning           1
                        deep learning              1
data scientist          machine learning          23
                        deep learning              9
                        nlp                        6
Name: count, dtype: int64
